In [1]:
import os, sys

# El kernel arranca desde notebooks/; mover al raíz del proyecto para que
# las rutas relativas de config.py (data/nba.sqlite, etc.) resuelvan correctamente.
project_root = os.path.abspath("..")
if os.getcwd() != project_root:
    os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from nba_predictor.storage import get_datastore
import pandas as pd

store = get_datastore()
games = store.load_games()
print(f"Total partidos: {len(games)}")
print(f"Temporadas: {sorted(games['season'].unique())}")
games.head()

Total partidos: 14429
Temporadas: ['2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']


,game_id,season,season_type,game_date,home_team_id,away_team_id,home_pts,away_pts,home_won,neutral_site
0,0021400003,2014-15,Regular Season,2014-10-28,1610612747,1610612745,90,108,0,0
1,0021400002,2014-15,Regular Season,2014-10-28,1610612759,1610612742,101,100,1,0
2,0021400001,2014-15,Regular Season,2014-10-28,1610612740,1610612753,101,84,1,0
3,0021400014,2014-15,Regular Season,2014-10-29,1610612758,1610612744,77,95,0,0
4,0021400007,2014-15,Regular Season,2014-10-29,1610612748,1610612764,107,95,1,0


In [3]:
ts = store.load_team_game_stats(season="2023-24")
gid = ts["game_id"].iloc[0]
sample = ts[ts["game_id"] == gid]
print("CHECK 1 — Rebotes del oponente:")
print(sample[["game_id", "team_id", "is_home", "oreb", "dreb", "fga", "tov"]])
print(f"  Filas para este partido: {len(sample)} (debe ser 2)\n")

CHECK 1 — Rebotes del oponente:
      game_id     team_id  is_home  oreb  dreb  fga  tov
0  0022300062  1610612756        0    17    43   95   19
1  0022300062  1610612744        1    18    31  101   11
  Filas para este partido: 2 (debe ser 2)



In [4]:
# CHECK PREVIO 2 — Orden cronológico sin ambigüedad (BLOQUEANTE)
print("CHECK 2 — Orden cronológico:")
print(f"  Fechas nulas: {games['game_date'].isna().sum()} (debe ser 0)")
# Cada partido aparece para 2 equipos; construimos (team, date) desde home y away
home = games[["home_team_id", "game_date"]].rename(columns={"home_team_id": "team_id"})
away = games[["away_team_id", "game_date"]].rename(columns={"away_team_id": "team_id"})
team_dates = pd.concat([home, away])
dups = team_dates.groupby(["team_id", "game_date"]).size()
print(f"  Pares (equipo, fecha) con >1 partido: {(dups > 1).sum()} (debe ser 0)\n")

CHECK 2 — Orden cronológico:
  Fechas nulas: 0 (debe ser 0)
  Pares (equipo, fecha) con >1 partido: 0 (debe ser 0)



In [6]:
print("CHECK 3 — Tabla teams:")
try:
    import sqlite3
    from nba_predictor.config import settings
    with sqlite3.connect(settings.db_path) as conn:
        n_teams = conn.execute("SELECT COUNT(*) FROM teams").fetchone()[0]
    print(f"  Equipos en tabla teams: {n_teams} (debe ser 30)\n")
except Exception as e:
    print(f"  Tabla teams no accesible o vacía: {e}\n")

# CHECK OPCIONAL — Realidad de los triples por temporada
print("CHECK OPCIONAL — Triples intentados por temporada (debe crecer):")
ts_all = store.load_team_game_stats()
merged = ts_all.merge(games[["game_id", "season"]], on="game_id")
print(merged.groupby("season")["fg3a"].mean().round(1))

CHECK 3 — Tabla teams:
  Equipos en tabla teams: 30 (debe ser 30)

CHECK OPCIONAL — Triples intentados por temporada (debe crecer):
season
2014-15    22.4
2015-16    24.1
2016-17    27.0
2017-18    29.0
2018-19    32.0
2019-20    34.1
2020-21    34.6
2021-22    35.2
2022-23    34.2
2023-24    35.1
2024-25    37.6
2025-26    37.0
Name: fg3a, dtype: float64
